# 🔬 Local Resolution Estimation in Cryo-EM 3D Maps

---

## A Note on the Dataset (please read first)

Real local resolution estimation needs two independently-refined
**half-maps** from a real single-particle reconstruction (typically
deposited alongside the final map in EMDB).

So, as in the other notebooks in this series, we build the two
half-maps ourselves from **real 3D EM density** — the same real
*Drosophila* ventral nerve cord volume (Cardona et al. 2010) used
throughout — by applying a **physically-motivated, spatially-varying
blur** (simulating a real, common scenario: a rigid, well-ordered core
surrounded by a more flexible, disordered periphery) and adding two
independent noise realizations. This gives us exact ground truth to
validate the local resolution algorithm against. Every number below is
real and measured.

## Overview

A single global FSC (Fourier Shell Correlation) resolution number —
the standard headline number reported for a cryo-EM structure — is an
*average* over the whole map. Real macromolecules are rarely uniformly
ordered: rigid cores reconstruct sharply, while flexible loops,
termini, or ligand-binding regions are smeared out and appear at much
lower *local* resolution, even though they contribute to the same
global FSC curve. **Local resolution estimation** (ResMap, MonoRes,
Blocres) recovers this spatial variation directly.

| Module | Topic |
|--------|-------|
| **1**  | Simulating Two Half-Maps with Real, Spatially-Varying Resolution |
| **2**  | Gold-Standard FSC & Why a Single Number Isn't Enough |
| **3**  | Implementation: Global FSC and Sliding-Window Local FSC |
| **4**  | Results: Recovered Local Resolution vs. Ground Truth |
| **5**  | Production Methods & Limitations |

> **Prerequisites:** `numpy`, `scipy`, `matplotlib`, `Pillow`.
> All cells are self-contained; the dataset auto-downloads on first run.

In [ ]:
# ============================================================
# GLOBAL IMPORTS & CONFIGURATION
# ============================================================
import os
import glob
import time
import warnings
import urllib.request

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from scipy import ndimage

warnings.filterwarnings("ignore")
np.random.seed(0)

DARK_BG, ACCENT, TEXT = "#0d1117", "#58a6ff", "#e6edf3"
plt.rcParams.update({
    "figure.facecolor": DARK_BG, "axes.facecolor": DARK_BG,
    "axes.edgecolor": TEXT, "axes.labelcolor": TEXT,
    "xtick.color": TEXT, "ytick.color": TEXT, "text.color": TEXT,
    "figure.titlesize": 14,
})

In [ ]:
# ============================================================
# MODULE 1 — REAL 3D VOLUME + SPATIALLY-VARYING RESOLUTION + TWO HALF-MAPS
# (Cardona et al. 2010, Drosophila VNC ssTEM — real volumetric density)
# ============================================================
DATA_DIR = "vnc_local_res"
os.makedirs(DATA_DIR, exist_ok=True)
BASE_URL = ("https://raw.githubusercontent.com/unidesigner/"
            "groundtruth-drosophila-vnc/master/stack1/raw")
N_SLICES = 20
for i in range(N_SLICES):
    fpath = os.path.join(DATA_DIR, f"{i:02d}.tif")
    if not os.path.exists(fpath):
        try:
            urllib.request.urlretrieve(f"{BASE_URL}/{i:02d}.tif", fpath)
        except Exception as e:
            print(f"  Warning: could not fetch slice {i:02d}: {e}")

raw_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.tif")))
volume = np.stack([np.array(Image.open(f)).astype(np.float32) / 255.0 for f in raw_files])
sub = volume[:, 400:528, 400:528]  # real (20, 128, 128) sub-volume
Z, Y, X = sub.shape
print(f"Real 3D EM sub-volume loaded: {sub.shape} (z, y, x)")

# --- define a real, physically-motivated spatially-varying "true" resolution:
#     a sharp, well-ordered core and a blurrier, flexible periphery ---
zz, yy, xx = np.mgrid[0:Z, 0:Y, 0:X]
cy, cx = Y / 2, X / 2
dist_xy = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2)
SIGMA_MIN, SIGMA_MAX = 0.4, 3.0
true_sigma_field = SIGMA_MIN + (SIGMA_MAX - SIGMA_MIN) * (dist_xy / dist_xy.max())

SIGMA_LEVELS = np.array([0.4, 1.0, 1.6, 2.2, 2.8, 3.4])
blurred_levels = [ndimage.gaussian_filter(sub, s) for s in SIGMA_LEVELS]


def spatially_varying_blur(vol, sigma_field, levels, precomputed):
    """Blend between precomputed fixed-sigma blurs to approximate a
    smoothly spatially-varying blur, efficiently."""
    out = np.zeros_like(vol)
    for i in range(len(levels) - 1):
        s0, s1 = levels[i], levels[i + 1]
        mask = (sigma_field >= s0) & (sigma_field < s1)
        if not mask.any():
            continue
        w = (sigma_field[mask] - s0) / (s1 - s0)
        out[mask] = (1 - w) * precomputed[i][mask] + w * precomputed[i + 1][mask]
    out[sigma_field < levels[0]] = precomputed[0][sigma_field < levels[0]]
    out[sigma_field >= levels[-1]] = precomputed[-1][sigma_field >= levels[-1]]
    return out


true_object = spatially_varying_blur(sub, true_sigma_field, SIGMA_LEVELS, blurred_levels)

# --- two independent noise realizations = two independent "half-maps" ---
rng = np.random.RandomState(0)
NOISE_STD = 0.06
half_A = true_object + rng.normal(0, NOISE_STD, true_object.shape).astype(np.float32)
half_B = true_object + rng.normal(0, NOISE_STD, true_object.shape).astype(np.float32)
print("Two independent half-maps generated from real density + known blur field.")

# ------------------------------------------------------------------
# VISUALIZATION 1 — The real structure, the true resolution field, and one half-map
# ------------------------------------------------------------------
z_mid = Z // 2
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle("Module 1 — Real Density, True Resolution Field, and a Simulated Half-Map",
             color=ACCENT, fontweight="bold")
axes[0].imshow(sub[z_mid], cmap="gray")
axes[0].set_title("Real EM density (sharp)", color=TEXT, fontsize=10)
im1 = axes[1].imshow(true_sigma_field[z_mid], cmap="inferno")
axes[1].set_title("True blur field\n(sharp core, blurry periphery)", color=TEXT, fontsize=10)
plt.colorbar(im1, ax=axes[1], fraction=0.046)
axes[2].imshow(half_A[z_mid], cmap="gray")
axes[2].set_title("Half-map A\n(blurred + independent noise)", color=TEXT, fontsize=10)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

---
# Module 2 — Gold-Standard FSC & Why a Single Number Isn't Enough

**Gold-standard FSC**: split the particle stack into two random halves,
reconstruct each independently, and correlate the two resulting maps
**per Fourier shell** (per spatial frequency). The Fourier Shell
Correlation is:

$$\text{FSC}(r) = \frac{\text{Re}\sum_{|\mathbf{k}|=r} F_A(\mathbf{k})\,F_B^*(\mathbf{k})}
{\sqrt{\sum_{|\mathbf{k}|=r}|F_A(\mathbf{k})|^2 \sum_{|\mathbf{k}|=r}|F_B(\mathbf{k})|^2}}$$

The reported "resolution" is the spatial frequency where FSC first
drops below **0.143** — the standard cryo-EM convention. This gives
**one number for the whole map**, which silently averages over
potentially very different local behaviors — exactly what Module 4
will make visible.

---
# Module 3 — Implementation: Global FSC and Sliding-Window Local FSC

**Local resolution** (the ResMap/Blocres approach): slide a small 3D
window across the volume, compute the FSC *only within that window*
between the two half-maps, and assign the resulting resolution value
to that window's location — repeat across the whole volume to build a
per-voxel resolution map.

In [ ]:
# ============================================================
# MODULE 3 — FSC AND LOCAL RESOLUTION MAP
# ============================================================
def radial_fsc(vol_a, vol_b):
    """FSC(shell) between two same-shape 3D volumes."""
    Fa = np.fft.fftshift(np.fft.fftn(vol_a))
    Fb = np.fft.fftshift(np.fft.fftn(vol_b))
    shape = vol_a.shape
    zz, yy, xx = np.mgrid[0:shape[0], 0:shape[1], 0:shape[2]]
    center = np.array(shape) / 2
    r = np.sqrt((zz - center[0]) ** 2 + (yy - center[1]) ** 2 + (xx - center[2]) ** 2).astype(int)
    max_r = r.max()
    fsc = np.zeros(max_r + 1)
    for radius in range(max_r + 1):
        mask = r == radius
        if mask.sum() < 3:
            fsc[radius] = np.nan
            continue
        a, b = Fa[mask], Fb[mask]
        num = np.real(np.sum(a * np.conj(b)))
        den = np.sqrt(np.sum(np.abs(a) ** 2) * np.sum(np.abs(b) ** 2))
        fsc[radius] = num / (den + 1e-10)
    return fsc


def resolution_from_fsc(fsc, threshold=0.143, box_size=None):
    box_size = box_size or len(fsc) * 2
    below = np.where(fsc < threshold)[0]
    shell = below[0] if len(below) > 0 else len(fsc) - 1
    shell = max(shell, 1)
    freq = shell / box_size  # cycles/voxel
    return 1.0 / freq if freq > 0 else np.inf


t0 = time.time()
fsc_global = radial_fsc(half_A, half_B)
res_global = resolution_from_fsc(fsc_global, box_size=max(sub.shape))
print(f"Global FSC computed in {time.time()-t0:.2f}s | "
      f"single global resolution estimate: {res_global:.1f} voxels")


def local_resolution_map(half_a, half_b, window=24, stride=8, threshold=0.143):
    Zc, Y, X = half_a.shape
    res_map = np.full((Zc, Y, X), np.nan)
    wz = min(window, Zc)
    hann_2d = np.outer(np.hanning(window), np.hanning(window))
    for y0 in range(0, Y - window + 1, stride):
        for x0 in range(0, X - window + 1, stride):
            wa = half_a[:wz, y0:y0 + window, x0:x0 + window] * hann_2d[None, :, :]
            wb = half_b[:wz, y0:y0 + window, x0:x0 + window] * hann_2d[None, :, :]
            fsc = radial_fsc(wa, wb)
            res = resolution_from_fsc(fsc, threshold=threshold, box_size=window)
            cy0, cy1 = y0 + window // 2 - stride // 2, y0 + window // 2 + stride // 2
            cx0, cx1 = x0 + window // 2 - stride // 2, x0 + window // 2 + stride // 2
            res_map[:, cy0:cy1, cx0:cx1] = res
    return res_map


t0 = time.time()
local_res = local_resolution_map(half_A, half_B, window=24, stride=8)
print(f"Local resolution map computed in {time.time()-t0:.2f}s | "
      f"range: {np.nanmin(local_res):.1f}-{np.nanmax(local_res):.1f} voxels")

---
# Module 4 — Results: Recovered Local Resolution vs. Ground Truth

Since we control the ground truth (the injected blur field), we can
directly check whether the recovered local resolution map actually
tracks the real spatial pattern — something impossible to verify on a
genuine unknown structure.

In [ ]:
# ============================================================
# MODULE 4 — VALIDATION AGAINST GROUND TRUTH
# ============================================================
res_slice = local_res[z_mid]
true_slice = true_sigma_field[z_mid]
valid = ~np.isnan(res_slice)
correlation = np.corrcoef(res_slice[valid], true_slice[valid])[0, 1]
print(f"Correlation between recovered local resolution and true injected blur: r = {correlation:.3f}")
print(f"(Global FSC alone reports one number, {res_global:.1f} voxels, for the entire volume — "
      f"masking this real spatial variation entirely.)")

# ------------------------------------------------------------------
# VISUALIZATION 2 — Side-by-side: true blur field vs. recovered local resolution
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Module 4 — Recovered Local Resolution Tracks the Real Ground-Truth Pattern",
             color=ACCENT, fontweight="bold")
axes[0].imshow(sub[z_mid], cmap="gray")
axes[0].set_title("Real EM density", color=TEXT, fontsize=10)
im1 = axes[1].imshow(true_slice, cmap="inferno")
axes[1].set_title("True blur field (ground truth)", color=TEXT, fontsize=10)
plt.colorbar(im1, ax=axes[1], fraction=0.046)
im2 = axes[2].imshow(res_slice, cmap="inferno")
axes[2].set_title(f"Recovered local resolution\n(r = {correlation:.2f} vs. ground truth)", color=TEXT, fontsize=10)
plt.colorbar(im2, ax=axes[2], fraction=0.046)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------
# VISUALIZATION 3 — Resolution vs. distance from the ordered core
# ------------------------------------------------------------------
dist_flat = dist_xy[z_mid][valid]
res_flat = res_slice[valid]
bins = np.linspace(dist_flat.min(), dist_flat.max(), 10)
bin_centers = (bins[:-1] + bins[1:]) / 2
binned_res = [res_flat[(dist_flat >= bins[i]) & (dist_flat < bins[i + 1])].mean() for i in range(len(bins) - 1)]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(bin_centers, binned_res, "o-", color=ACCENT, linewidth=2)
ax.set_xlabel("Distance from center (voxels)")
ax.set_ylabel("Recovered local resolution (voxels; lower = better)")
ax.set_title("Module 4 — Resolution Degrades Away from the Ordered Core",
             color=ACCENT, fontweight="bold")
plt.tight_layout()
plt.show()

---
# Module 5 — Production Methods & Limitations

| Method | Approach | Notes |
|--------|----------|-------|
| ResMap (Kucukelbir et al. 2014) | Sliding-window local FSC via local structure tensor | The direct ancestor of Module 3's approach |
| Blocres (Cardone et al. 2013, Bsoft) | Block-based local FSC | Similar sliding-window design |
| MonoRes (Vilas et al. 2018) | Monogenic-signal-based, single-map local resolution | Doesn't require two independent half-maps |
| cryoSPARC Local Resolution | Windowed FSC integrated into the refinement pipeline | Now a standard automatic output of modern pipelines |
| DeepRes (Ramirez-Aportela et al. 2019) | Deep-learning-based local resolution estimation | Learns to predict local resolution directly from map features |

## Known Limitations of This Tutorial
- **Not real half-maps** (see the note at the top) — two independent
  noise realizations were added to a real, deliberately blurred
  density map to create controllable ground truth.
- **Isotropic blur as a stand-in for "local resolution"**: this
  notebook uses real-space Gaussian blur as a proxy for locally
  degraded resolution; real local resolution variation more often
  arises from anisotropic particle-orientation coverage and genuine
  conformational flexibility, not simple isotropic blurring.
- **Small window size** (24 voxels here) limits the achievable
  frequency resolution of each local FSC curve — a fundamental
  trade-off between spatial and frequency resolution that real tools
  navigate the same way.
- **Block artifacts**: the sliding-window map has visible blocky
  transitions at this stride; production tools use finer strides
  and/or smoother interpolation.

In [ ]:
# ============================================================
# FINAL DASHBOARD — Complete pipeline summary
# ============================================================
fig = plt.figure(figsize=(20, 11))
fig.patch.set_facecolor(DARK_BG)
fig.suptitle("🔬 Local Resolution Estimation — Pipeline Dashboard",
             fontsize=15, fontweight="bold", color=ACCENT, y=0.98)
gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.5, wspace=0.3)

ax0 = fig.add_subplot(gs[0, 0]); ax0.imshow(sub[z_mid], cmap="gray"); ax0.set_title("Real EM density", color=TEXT, fontsize=10); ax0.axis("off")
ax1 = fig.add_subplot(gs[0, 1]); im1 = ax1.imshow(true_slice, cmap="inferno"); ax1.set_title("True blur field", color=TEXT, fontsize=10); ax1.axis("off")
ax2 = fig.add_subplot(gs[0, 2]); im2 = ax2.imshow(res_slice, cmap="inferno"); ax2.set_title("Recovered local resolution", color=TEXT, fontsize=10); ax2.axis("off")
ax3 = fig.add_subplot(gs[0, 3])
ax3.plot(bin_centers, binned_res, "o-", color=ACCENT, linewidth=2)
ax3.set_title("Resolution vs. distance from core", color=TEXT, fontsize=9)
ax3.set_xlabel("distance (voxels)", fontsize=8)

ax4 = fig.add_subplot(gs[1, 0:2])
shells = np.arange(len(fsc_global))
ax4.plot(shells, fsc_global, color=ACCENT, linewidth=2)
ax4.axhline(0.143, color="#f0883e", linestyle="--", label="0.143 threshold")
ax4.set_xlabel("Fourier shell"); ax4.set_ylabel("FSC")
ax4.set_title(f"Global FSC curve (single resolution = {res_global:.1f} voxels)", color=TEXT, fontsize=10)
ax4.legend(facecolor=DARK_BG, labelcolor=TEXT, fontsize=8)

ax5 = fig.add_subplot(gs[1, 2:4])
ax5.text(0.05, 0.7, f"Correlation (recovered vs. true): r = {correlation:.3f}", color=TEXT, fontsize=13, transform=ax5.transAxes)
ax5.text(0.05, 0.5, f"Global FSC resolution: {res_global:.1f} voxels (one number, whole volume)", color=TEXT, fontsize=13, transform=ax5.transAxes)
ax5.text(0.05, 0.3, f"Local resolution range: {np.nanmin(local_res):.1f}-{np.nanmax(local_res):.1f} voxels", color=TEXT, fontsize=13, transform=ax5.transAxes)
ax5.axis("off")

plt.tight_layout()
plt.show()

---
# Summary

## What This Notebook Demonstrated

| Step | Module | Key Idea |
|------|--------|----------|
| Data honesty | — | Real half-maps weren't reachable; simulated two independent noise realizations of a real, deliberately spatially-blurred density map |
| Problem framing | 2 | A single global FSC number averages over real, meaningful local variation |
| Implementation | 3 | Global FSC and sliding-window local FSC, both from first principles |
| Results | 4 | Real measured correlation between recovered and true local resolution; resolution degrades with distance from the ordered core, as expected |
| Context | 5 | Positioned against ResMap, Blocres, MonoRes, cryoSPARC, DeepRes |

## Computational Complexity

| Step | Complexity | Bottleneck |
|------|------------|------------|
| Global FSC | $\mathcal{O}(N^3 \log N)$ | One 3D FFT per half-map |
| Local FSC (per window) | $\mathcal{O}(W^3 \log W)$ | $W$ = window size; small and fast |
| Local resolution map (full volume) | $\mathcal{O}\!\left(\frac{N^2}{s^2}\cdot W^3\log W\right)$ | $s$ = stride; dominant cost, still fast at this volume size |

## Key References
- Kucukelbir, Sigworth & Tagare (2014) — ResMap: local resolution estimation (*Nature Methods*)
- Cardone, Heymann & Steven (2013) — Blocres local resolution (*J. Struct. Biol.*)
- Vilas et al. (2018) — MonoRes: single-map local resolution via the monogenic signal (*Structure*)
- Ramirez-Aportela et al. (2019) — DeepRes: deep-learning local resolution estimation (*Bioinformatics*)
- Rosenthal & Henderson (2003) — Gold-standard FSC and the 0.143 criterion (*J. Mol. Biol.*)
- Cardona et al. (2010) — ssTEM Drosophila VNC dataset used as the real volumetric source (*PLoS Biology*)